In [39]:
import os
import sys
import json
import glob
import pickle
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.BigSmall import BigSmall

In [40]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/bigsmall")

# 1. Khai báo MODEL_PATH trước
MODEL_PATH          = os.path.join(REPO_ROOT, "final_model_release", "BP4D_BigSmall_Multitask_Fold3.pth") # 1 or 2 or 3

# 2. Lấy tên model từ file path (VD: "BP4D_BigSmall_Multitask_Fold1")
MODEL_NAME = os.path.basename(MODEL_PATH).replace(".pth", "")

# 3. Nối tên model vào thư mục OUTPUT_DIR
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Headmotion/bigsmall", MODEL_NAME)

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- BigSmall preprocessing params -----
CHUNK_LENGTH = 3       # frames per clip (same as training config)
BIG_H, BIG_W = 144, 144
SMALL_H, SMALL_W = 9, 9

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----- 49-channel label layout (matches BigSmallTrainer label_list) -----
LABEL_IDX_BVP  = 0   # bp_wave (green-channel PPG signal)
LABEL_IDX_HR   = 1   # HR_bpm
LABEL_IDX_RESP = 5   # resp_wave
NUM_LABEL_CH   = 49

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall
OUTPUT_DIR: /home/iec/MinhHieu/rPPG/results/Headmotion/bigsmall/BP4D_BigSmall_Multitask_Fold3


In [41]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [42]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [43]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [44]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [45]:
# Resize all frames (no face detection)

def resize_frames(frames, out_h, out_w):
    """Resize every frame to (out_h, out_w) using INTER_AREA."""
    T, H, W, C = frames.shape
    resized = np.zeros((T, out_h, out_w, C), dtype=np.float32)
    for i in range(T):
        resized[i] = cv2.resize(frames[i], (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [46]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 10 subject folders

  S_001  video=S_001.mkv
  S_002  video=S_002.mkv
  S_003  video=S_003.mkv
  S_004  video=S_004.mkv
  S_005  video=S_005.mkv
  S_006  video=S_006.mkv
  S_007  video=S_007.mkv
  S_008  video=S_008.mkv
  S_009  video=S_009.mkv
  S_010  video=S_010.mkv

Total subjects: 10


In [47]:
# Data preprocessing for BigSmall / Multitask models

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # 1. Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # 2. Read and resample PPG (Dùng hàm mới đã sửa)
    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # 3. Prepare 49-channel label array
    # Khởi tạo tất cả là -1.0
    labels = np.full((T, NUM_LABEL_CH), -1.0, dtype=np.float32)
    
    # Gán tín hiệu PPG thô vào kênh BVP trước khi chuẩn hóa (tùy chọn, để kiểm tra)
    labels[:, LABEL_IDX_BVP] = ppg_signal
    # ĐÃ XÓA: labels[:, LABEL_IDX_HR] = avg_hr (Để mặc định là -1.0)

    # 4. Multi-scale Face Processing (Big & Small)
    frames_big = crop_face_resize(frames, BIG_H, BIG_W)

    big_data   = standardized_data(frames_big)
    diff_big   = diff_normalize_data(frames_big)
    # Tạo nhánh dữ liệu nhỏ (thường dùng cho các model như BigSmall)
    small_data = resize_frames(diff_big, SMALL_H, SMALL_W)

    # 5. Normalize Label
    # Chuẩn hóa tín hiệu PPG (DiffNormalized) để làm nhãn học máy
    labels[:, LABEL_IDX_BVP] = diff_normalize_label(ppg_signal)

    # 6. Chunk into clips
    clip_num    = T // CHUNK_LENGTH
    big_clips   = np.array([big_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]   for i in range(clip_num)])
    small_clips = np.array([small_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([labels[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]     for i in range(clip_num)])

    # 7. Save to disk (.pickle cho input và .npy cho label)
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.pickle")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        # Lưu dictionary chứa cả 2 scale dữ liệu
        frames_dict = {0: big_clips[chunk_idx], 1: small_clips[chunk_idx]}
        with open(input_path, "wb") as fh:
            pickle.dump(frames_dict, fh, protocol=pickle.HIGHEST_PROTOCOL)
        
        np.save(label_path, label_clips[chunk_idx])
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*.pickle")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall

=== Processing S001 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=91
  901 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall/S001

=== Processing S002 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=94
  901 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall/S002

=== Processing S003 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=2, max=95
  900 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall/S003

=== Processing S004 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=107
  900 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall/S004

=== Processing S005 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=117
  900 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/bigsmall/S005

=== Processing S006 ===
  Video: 2702 frames @ 30 fps
  PPG green: min=1, max=105
  900 clip

In [48]:
# PyTorch Dataset

class BigSmallDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label").replace(".pickle", ".npy")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        with open(self.inputs[index], "rb") as fh:
            data = pickle.load(fh)

        # NDHWC -> NDCHW
        data[0] = np.float32(np.transpose(data[0], (0, 3, 1, 2)))
        data[1] = np.float32(np.transpose(data[1], (0, 3, 1, 2)))

        label = np.float32(np.load(self.labels[index]))  # (D, 49)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = BigSmallDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

Dataset: 9003 clips


In [49]:
# Custom collate function + DataLoader
0
def bigsmall_collate(batch):
    """Stack dict-based data into tensors.

    Returns:
        data_big   (Tensor): (N, D, C, H_big, W_big)
        data_small (Tensor): (N, D, C, H_small, W_small)
        labels     (Tensor): (N, D, 49)
        subjects   (list[str])
        chunk_ids  (list[str])
    """
    data_dicts, labels, subjects, chunk_ids = zip(*batch)

    data_big   = torch.stack([torch.from_numpy(d[0]) for d in data_dicts], dim=0)
    data_small = torch.stack([torch.from_numpy(d[1]) for d in data_dicts], dim=0)
    labels_t   = torch.stack([torch.from_numpy(l)    for l in labels],     dim=0)

    return data_big, data_small, labels_t, list(subjects), list(chunk_ids)


loader = DataLoader(dataset, batch_size=32, shuffle=False,
                    num_workers=4, collate_fn=bigsmall_collate)
print(f"DataLoader ready: {len(loader)} batches")

DataLoader ready: 282 batches


In [50]:
# Load pretrained BigSmall model

model = BigSmall(n_segment=CHUNK_LENGTH)

state_dict = torch.load(MODEL_PATH, map_location=DEVICE)

# strip 'module.' prefix if present (DataParallel artifact)
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print("Model loaded and set to eval mode.")
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

/tmp/ipykernel_3190528/371294202.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(MODEL_PATH, map_location=DEVICE)


Model loaded and set to eval mode.
Total parameters: 2,142,478


In [51]:
# Inference 

BASE_LEN = CHUNK_LENGTH  # num_gpu=1

# BVP
bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
bvp_labels_dict = {}

# RESP
resp_preds_dict = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}

with torch.no_grad():
    for batch in tqdm(loader, desc="Inference"):
        data_big, data_small, labels_t, batch_subjects, batch_chunk_ids = batch

        N, D, C_b, H_b, W_b = data_big.shape
        N, D, C_s, H_s, W_s = data_small.shape

        # flatten temporal dimension
        big_flat   = data_big.view(N * D, C_b, H_b, W_b).to(DEVICE)
        small_flat = data_small.view(N * D, C_s, H_s, W_s).to(DEVICE)

        # trim to multiple of BASE_LEN for TSM alignment
        trim = (N * D) // BASE_LEN * BASE_LEN
        big_flat   = big_flat[:trim]
        small_flat = small_flat[:trim]

        labels_flat = labels_t.view(N * D, NUM_LABEL_CH)  # (N*D, 49)

        au_out, bvp_out, resp_out = model((big_flat, small_flat))

        bvp_pred_np  = bvp_out.squeeze(-1).cpu().numpy()   # (trim,)
        resp_pred_np = resp_out.squeeze(-1).cpu().numpy()  # (trim,)
        bvp_label_np = labels_flat[:trim, LABEL_IDX_BVP].numpy()

        for i in range(N):
            if i * CHUNK_LENGTH >= trim:
                break

            subj  = batch_subjects[i]
            cid   = int(batch_chunk_ids[i])
            start = i * CHUNK_LENGTH
            end   = start + CHUNK_LENGTH

            if subj not in bvp_preds_dict:
                bvp_preds_dict[subj]  = {}
                bvp_labels_dict[subj] = {}
                resp_preds_dict[subj] = {}

            bvp_preds_dict[subj][cid]  = bvp_pred_np[start:end]
            bvp_labels_dict[subj][cid] = bvp_label_np[start:end]
            resp_preds_dict[subj][cid] = resp_pred_np[start:end]

print("\nInference complete.")
print("Subjects:", sorted(bvp_preds_dict.keys()))

Inference: 100%|██████████| 282/282 [03:53<00:00,  1.21it/s]


Inference complete.
Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']


In [52]:
# Post-processing

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])

# Per-subject BVP post-processing

def process_bvp(pred_chunks, label_chunks, fs=30):
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    pred  = detrend(np.cumsum(pred),  100)
    label = detrend(np.cumsum(label), 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

# Per-subject RESP post-processing

def process_resp(pred_chunks, fs=30):
    pred = _reform_from_dict(pred_chunks).astype(np.float64)

    pred = detrend(np.cumsum(pred), 100)
    pred = bandpass_filter(pred, fs, low=0.13, high=0.5)

    rr_pred_bpm = fft_peak_hz(pred, fs, 0.13, 0.5) * 60.0
    return rr_pred_bpm

In [53]:
# Compute per-subject results

FS = VIDEO_FPS

per_subject_results = []

hr_preds_all  = []
hr_labels_all = []
snr_all       = []
rr_preds_all  = []

print(f"{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'RR_pred':>8} {'SNR':>7}")
print("-" * 62)

for subj_key in sorted(bvp_preds_dict.keys()):
    hr_pred, hr_label, _, pred_processed = process_bvp(
        bvp_preds_dict[subj_key], bvp_labels_dict[subj_key], fs=FS
    )
    rr_pred = process_resp(resp_preds_dict[subj_key], fs=FS)

    snr_db = calculate_snr(pred_processed, hr_label, FS)
    hr_err = hr_pred - hr_label

    # map subj_key ("S000") back to original ID ("S_000")
    subj_id = subj_key[0] + "_" + subj_key[1:]

    per_subject_results.append({
        "name":                subj_id,
        "predicted_heartrate": hr_pred,
        "label_heartrate":     hr_label,
        "heartrate_error":     hr_err,
        "respiration_rate":    rr_pred,
        "snr_db":              snr_db,
    })

    hr_preds_all.append(hr_pred)
    hr_labels_all.append(hr_label)
    snr_all.append(snr_db)
    rr_preds_all.append(rr_pred)

    print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {rr_pred:>8.3f} {snr_db:>7.2f}")

hr_preds_all  = np.array(hr_preds_all)
hr_labels_all = np.array(hr_labels_all)
snr_all       = np.array(snr_all)

Subject       HR_pred   HR_label   HR_err  RR_pred     SNR
--------------------------------------------------------------
S_001          50.977     85.693  -34.717   24.170   -5.33
S_002          86.572     86.572    0.000   23.730   -0.52
S_003          79.980     76.025    3.955   25.049   -0.97
S_004          96.240     97.559   -1.318   26.367   -6.36
S_005          89.648     87.451    2.197   21.533   -0.31
S_006          61.963     61.963    0.000   28.125   -2.30
S_007          96.680     94.922    1.758   20.215   -2.46
S_008          56.689     56.689    0.000   25.928    2.61
S_009          74.268     73.389    0.879   21.973   -3.71
S_010          89.209     89.209    0.000   16.699   -3.53


In [54]:
# Aggregate metrics

n = len(hr_preds_all)
assert n > 0, "No subjects to evaluate."

err   = hr_preds_all - hr_labels_all
abs_e = np.abs(err)
sq_e  = err ** 2
rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

mae       = float(np.mean(abs_e))
mae_se    = float(np.std(abs_e) / np.sqrt(n))

rmse      = float(np.sqrt(np.mean(sq_e)))
rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))

mape      = float(np.mean(rel_e) * 100.0)
mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

if n >= 2:
    pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
    pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
else:
    pearson_r, pearson_se = float("nan"), float("nan")

mean_snr    = float(np.mean(snr_all))
mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

print(f"MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
print(f"RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
print(f"MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
print(f"Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
print(f"SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

MAE     : 4.4824 +/- 3.2100 bpm
RMSE    : 11.0965 +/- 10.6806 bpm
MAPE    : 5.2629 +/- 3.7475 %
Pearson : 0.7348 +/- 0.2398
SNR     : -2.2884 +/- 0.7886 dB


In [55]:
# Export results

model_name = os.path.basename(MODEL_PATH).replace(".pth", "")

metrics_dict = {
    "model":      model_name,
    "n_subjects": n,
    "evaluation_method": "FFT BVP-derived HR",
    "bvp_bandpass_hz":   [0.6, 3.3],
    "resp_bandpass_hz":  [0.13, 0.5],
    "aggregate_metrics": {
        "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
        "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
        "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
        "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
        "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
    },
    "per_subject": [
        {
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "respiration_rate":    r["respiration_rate"],
            "snr_db":              r["snr_db"],
        }
        for r in per_subject_results
    ],
}

json_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(json_path, "w") as fh:
    json.dump(metrics_dict, fh, indent=2)

print(f"Metrics saved to: {json_path}")

csv_rows = []
for r in per_subject_results:
    csv_rows.append({
        "name":                r["name"],
        "predicted_heartrate": r["predicted_heartrate"],
        "label_heartrate":     r["label_heartrate"],
        "heartrate_error":     r["heartrate_error"],
        "respiration_rate":    r["respiration_rate"]
    })

results_df = pd.DataFrame(csv_rows, columns=[
    "name", "predicted_heartrate", "label_heartrate",
    "heartrate_error", "respiration_rate"
])

csv_path = os.path.join(OUTPUT_DIR, "ppg_results.csv")
results_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")
print()
print(results_df.to_string(index=False))

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/bigsmall/BP4D_BigSmall_Multitask_Fold3/metrics.json
CSV saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/bigsmall/BP4D_BigSmall_Multitask_Fold3/ppg_results.csv

 name  predicted_heartrate  label_heartrate  heartrate_error  respiration_rate
S_001            50.976562        85.693359       -34.716797         24.169922
S_002            86.572266        86.572266         0.000000         23.730469
S_003            79.980469        76.025391         3.955078         25.048828
S_004            96.240234        97.558594        -1.318359         26.367188
S_005            89.648438        87.451172         2.197266         21.533203
S_006            61.962891        61.962891         0.000000         28.125000
S_007            96.679688        94.921875         1.757812         20.214844
S_008            56.689453        56.689453         0.000000         25.927734
S_009            74.267578        73.388672         0.878906   

In [56]:
import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc (Thêm dấu / ở đầu để thành đường dẫn tuyệt đối)
ROOT_DIR = "/home/iec/MinhHieu/rPPG/results/Headmotion/bigsmall"  

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con (Fold1, Fold2, Fold3...)
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        # Lấy tên model ưu tiên từ JSON. Nếu không có hoặc chung chung, lấy theo tên thư mục (VD: BP4D_BigSmall_Multitask_Fold1)
        model_name = data.get("model")
        if not model_name or model_name == "Unknown" or model_name == "bigsmall":
            # os.path.dirname(file_path) lấy thư mục chứa file json
            # os.path.basename() lấy tên của thư mục đó
            model_name = os.path.basename(os.path.dirname(file_path))

        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV tổng (nằm ngang hàng với các thư mục Fold)
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")

✅ Đã gom thành công 3 file JSON!
✅ File tổng hợp được lưu tại:
/home/iec/MinhHieu/rPPG/results/Headmotion/bigsmall/Model_Performance_Metrics.csv

Preview dữ liệu:
 rank                         model  # subjects      MAE (bpm)      RMSE (bpm)       MAPE (%)       Pearson       SNR (dB)
    1 BP4D_BigSmall_Multitask_Fold3          10  4.48 +/- 3.21 11.10 +/- 10.68  5.26 +/- 3.75 0.73 +/- 0.24 -2.29 +/- 0.79
    2 BP4D_BigSmall_Multitask_Fold1          10  8.22 +/- 4.64 16.81 +/- 13.39  9.10 +/- 5.03 0.42 +/- 0.32 -3.46 +/- 0.76
    3 BP4D_BigSmall_Multitask_Fold2          10 10.77 +/- 5.31 19.96 +/- 15.57 12.23 +/- 5.92 0.38 +/- 0.33 -2.93 +/- 0.84
